# 01 - Preprocessing Data
**Proyek WASPADA** — Sistem Deteksi Fraud Transaksi Keuangan

Tahapan:
1. Load dataset
2. Eksplorasi data (EDA)
3. Normalisasi fitur
4. Handle imbalance dengan SMOTE
5. Split data train/test
6. Simpan hasil preprocessing

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import os

print('Semua library berhasil diimport!')

## 2. Load Dataset

In [ ]:
# Pastikan file creditcard.csv ada di folder notebook/
df = pd.read_csv('creditcard.csv')

print(f'Jumlah baris   : {df.shape[0]:,}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'\nKolom dataset  : {list(df.columns)}')

## 3. Eksplorasi Data (EDA)

In [ ]:
# Cek missing values
print('=== Missing Values ===')
print(df.isnull().sum().sum(), 'missing value ditemukan')

# Statistik dasar
print('\n=== Statistik Amount ===')
print(df['Amount'].describe())

In [ ]:
# Distribusi kelas (Normal vs Fraud)
kelas = df['Class'].value_counts()
print('=== Distribusi Kelas ===')
print(f'Normal (0) : {kelas[0]:,} transaksi ({kelas[0]/len(df)*100:.2f}%)')
print(f'Fraud  (1) : {kelas[1]:,} transaksi ({kelas[1]/len(df)*100:.2f}%)')

# Visualisasi distribusi kelas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal', 'Fraud'], [kelas[0], kelas[1]], color=['#1D9E75', '#D85A30'])
axes[0].set_title('Distribusi Kelas (jumlah)')
axes[0].set_ylabel('Jumlah Transaksi')
for i, v in enumerate([kelas[0], kelas[1]]):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=10)

axes[1].pie([kelas[0], kelas[1]], labels=['Normal', 'Fraud'],
            autopct='%1.2f%%', colors=['#1D9E75', '#D85A30'], startangle=90)
axes[1].set_title('Distribusi Kelas (%)')

plt.tight_layout()
plt.savefig('eda_distribusi_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: eda_distribusi_kelas.png')

In [ ]:
# Distribusi Amount: Normal vs Fraud
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[df['Class'] == 0]['Amount'].plot(kind='hist', bins=50, ax=axes[0],
    color='#1D9E75', alpha=0.7, title='Distribusi Amount - Normal')
axes[0].set_xlabel('Amount')

df[df['Class'] == 1]['Amount'].plot(kind='hist', bins=50, ax=axes[1],
    color='#D85A30', alpha=0.7, title='Distribusi Amount - Fraud')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.savefig('eda_distribusi_amount.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: eda_distribusi_amount.png')

## 4. Normalisasi Fitur Amount & Time

In [ ]:
# Kolom V1-V28 sudah dinormalisasi oleh PCA dari dataset aslinya
# Kita hanya perlu normalisasi kolom Amount dan Time

scaler_amount = StandardScaler()
scaler_time   = StandardScaler()

df['Amount_scaled'] = scaler_amount.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler_time.fit_transform(df[['Time']])

# Hapus kolom asli yang sudah diganti
df = df.drop(columns=['Amount', 'Time'])

print('Normalisasi selesai!')
print(f'Kolom sekarang: {list(df.columns)}')

## 5. Pisahkan Fitur (X) dan Label (y)

In [ ]:
X = df.drop(columns=['Class']).values
y = df['Class'].values

print(f'Shape X : {X.shape}')
print(f'Shape y : {y.shape}')
print(f'Jumlah fitur input ANN: {X.shape[1]}')

## 6. Split Data Train & Test (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data train  : {X_train.shape[0]:,} baris')
print(f'Data test   : {X_test.shape[0]:,} baris')
print(f'\nDistribusi train - Normal: {sum(y_train==0):,} | Fraud: {sum(y_train==1):,}')
print(f'Distribusi test  - Normal: {sum(y_test==0):,}  | Fraud: {sum(y_test==1):,}')

## 7. Handle Imbalance dengan SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) membuat data fraud sintetis
sehingga jumlah data fraud dan normal menjadi seimbang di data train.

> **Penting:** SMOTE hanya diterapkan pada data TRAIN, bukan data TEST!
> Data test harus tetap mencerminkan kondisi nyata.

In [ ]:
print('Sebelum SMOTE:')
print(f'  Normal: {sum(y_train==0):,} | Fraud: {sum(y_train==1):,}')

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nSetelah SMOTE:')
print(f'  Normal: {sum(y_train_sm==0):,} | Fraud: {sum(y_train_sm==1):,}')
print(f'  Total  : {len(y_train_sm):,} baris')

# Visualisasi sebelum vs sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Normal', 'Fraud'], [sum(y_train==0), sum(y_train==1)],
            color=['#1D9E75', '#D85A30'])
axes[0].set_title('Sebelum SMOTE')
axes[0].set_ylabel('Jumlah')

axes[1].bar(['Normal', 'Fraud'], [sum(y_train_sm==0), sum(y_train_sm==1)],
            color=['#1D9E75', '#D85A30'])
axes[1].set_title('Sesudah SMOTE')

plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: smote_comparison.png')

## 8. Simpan Hasil Preprocessing

In [ ]:
# Buat folder processed jika belum ada
os.makedirs('processed', exist_ok=True)

# Simpan data
np.save('processed/X_train.npy', X_train_sm)
np.save('processed/y_train.npy', y_train_sm)
np.save('processed/X_test.npy',  X_test)
np.save('processed/y_test.npy',  y_test)

# Simpan scaler (dibutuhkan saat prediksi di backend)
os.makedirs('../backend/model', exist_ok=True)
joblib.dump(scaler_amount, '../backend/model/scaler_amount.pkl')
joblib.dump(scaler_time,   '../backend/model/scaler_time.pkl')

print('Semua file berhasil disimpan!')
print('\nFile tersimpan:')
print('  notebook/processed/X_train.npy')
print('  notebook/processed/y_train.npy')
print('  notebook/processed/X_test.npy')
print('  notebook/processed/y_test.npy')
print('  backend/model/scaler_amount.pkl')
print('  backend/model/scaler_time.pkl')

## Ringkasan Preprocessing

| Tahap | Keterangan |
|---|---|
| Dataset | 284.807 transaksi, 30 fitur |
| Missing values | Tidak ada |
| Normalisasi | StandardScaler pada Amount & Time |
| Split | 80% train, 20% test (stratified) |
| SMOTE | Diterapkan hanya pada data train |
| Output | X_train, y_train, X_test, y_test + scaler |